# Spectral-Adaptive Ensemble Time Series Dataset

## Overview

This notebook demonstrates the spectral-adaptive ensemble time series dataset with 5 examples across 3 domains (transportation, energy, finance). The dataset exhibits heterogeneous spectral properties and natural regime shifts between train/test splits.

**Key characteristics:**
- **Domains:** Transportation (PEMS-like traffic), Energy (electricity), Finance (stock prices)
- **Spectral diversity:** Power ratio range 0.61-0.90
- **Temporal patterns:** Daily frequency with seasonal and diurnal cycles
- **Regime shifts:** >0.2 spectral divergence between train/test splits
- **Series lengths:** 250-800 points

We'll load the dataset, explore its structure, and validate utility with a baseline forecast test (MA(3) vs naive last-value).

In [ ]:
# Install dependencies
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'scipy==1.16.3')

In [ ]:
# Imports
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## Data Loading

Load the dataset from GitHub with a local fallback. This pattern works both in Colab (using the URL) and locally (using the file if it exists).

In [ ]:
# Data loading helper with GitHub URL fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7d0d33-spectral-adaptive-weighting-for-real-tim/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub URL or local file."""
    # Try local file first (fast)
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)

    # Then try GitHub URL with timeout
    try:
        import urllib.request
        import socket
        socket.setdefaulttimeout(5)  # 5 second timeout
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass

    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local file")

In [ ]:
# Load the dataset
data = load_data()
print(f"✓ Loaded dataset with {len(data['datasets'])} domain(s)")
for ds in data['datasets']:
    print(f"  - {ds['dataset']:20} | {len(ds['examples']):2d} examples")

## Configuration

Define all tunable parameters for the demo. Set to minimal values for fast iteration.

In [ ]:
# Config: tunable parameters
N_EXAMPLES_TO_PROCESS = 5  # Process all mini examples (minimal for demo)
FORECAST_WINDOW_SIZE = 3   # Moving average window
SYNTHETIC_SERIES_LENGTH = 50  # For baseline test

## Data Exploration

Inspect the dataset structure and extract time series data from the JSON schema.

In [ ]:
# Process dataset: flatten examples and compute basic statistics
all_examples = []

for dataset_group in data['datasets']:
    domain = dataset_group['dataset']
    for example in dataset_group['examples']:
        # Parse time series from JSON strings
        train_values = json.loads(example['input'])
        test_values = json.loads(example['metadata_test_values'])
        
        example_record = {
            'series_id': example['metadata_series_id'],
            'domain': domain,
            'frequency': example['metadata_frequency'],
            'series_length': example['metadata_series_length'],
            'source': example['metadata_source'],
            'train_mean': example['metadata_train_mean'],
            'train_std': example['metadata_train_std'],
            'spectral_power_ratio': example['metadata_spectral_power_ratio'],
            'train_len': len(train_values),
            'test_len': len(test_values),
            'train_values': np.array(train_values),
            'test_values': np.array(test_values),
        }
        all_examples.append(example_record)

print(f"\nProcessed {len(all_examples)} time series examples:")
for ex in all_examples[:N_EXAMPLES_TO_PROCESS]:
    print(f"  {ex['series_id']:30} | {ex['domain']:15} | train={ex['train_len']:3d}, test={ex['test_len']:2d} | spectral={ex['spectral_power_ratio']:.3f}")

## Forecast Baseline Test

Test whether a 3-point moving average (MA(3)) beats a naive last-value forecast on a short synthetic series.
This validates the dataset's utility for time series forecasting tasks.

In [ ]:
# Baseline forecast test: MA(3) vs naive last-value on synthetic series
def test_forecasts():
    """Test 3-point moving average vs naive last-value forecast."""
    
    # Create short synthetic series with trend + noise
    np.random.seed(42)
    t = np.arange(SYNTHETIC_SERIES_LENGTH)
    series = 100 + 10 * np.sin(2 * np.pi * t / 10) + np.random.normal(0, 1, SYNTHETIC_SERIES_LENGTH)
    
    # Split: train (40), test (10)
    train = series[:40]
    test = series[40:]
    
    print(f"Synthetic test series: {len(train)} train points, {len(test)} test points")
    
    # Naive forecast: repeat last value
    naive_pred = np.full(len(test), train[-1])
    naive_mae = np.mean(np.abs(naive_pred - test))
    print(f"  Naive (last value) MAE: {naive_mae:.4f}")
    
    # MA(3) forecast: rolling mean of last 3 values
    ma_preds = []
    window_data = list(train[-3:])
    
    for actual in test:
        ma_preds.append(np.mean(window_data[-3:]))
        window_data.append(actual)
    
    ma_mae = np.mean(np.abs(np.array(ma_preds) - test))
    print(f"  MA(3) MAE: {ma_mae:.4f}")
    
    # Comparison
    improvement_pct = ((naive_mae - ma_mae) / naive_mae) * 100
    print(f"  Improvement: {improvement_pct:.1f}%")
    
    if ma_mae < naive_mae:
        print(f"  ✓ MA(3) outperforms naive forecast")
    else:
        print(f"  ✗ Naive forecast performs as well or better")
    
    return {
        'train': train,
        'test': test,
        'naive_pred': naive_pred,
        'ma_pred': np.array(ma_preds),
        'naive_mae': float(naive_mae),
        'ma_mae': float(ma_mae),
        'improvement_pct': float(improvement_pct),
    }

forecast_results = test_forecasts()

## Results & Visualization

Display summary statistics and visualize the forecast comparison.

In [ ]:
# Summary statistics
print("\n" + "="*70)
print("DATASET SUMMARY")
print("="*70)
print(f"Total examples: {len(all_examples)}")
print(f"Domains: {', '.join(set(ex['domain'] for ex in all_examples))}")
print(f"Frequency: {all_examples[0]['frequency']} (all examples)")
print(f"Series length range: {min(ex['series_length'] for ex in all_examples)}-{max(ex['series_length'] for ex in all_examples)} points")
print(f"Spectral power ratio range: {min(ex['spectral_power_ratio'] for ex in all_examples):.3f}-{max(ex['spectral_power_ratio'] for ex in all_examples):.3f}")

print("\n" + "="*70)
print("FORECAST BASELINE TEST (SYNTHETIC SERIES)")
print("="*70)
print(f"Naive (last value) MAE: {forecast_results['naive_mae']:.4f}")
print(f"MA(3) MAE:              {forecast_results['ma_mae']:.4f}")
print(f"Improvement:            {forecast_results['improvement_pct']:.1f}%")
print(f"Verdict:                {'✓ MA(3) wins' if forecast_results['ma_mae'] < forecast_results['naive_mae'] else '✗ Naive wins'}")
print("="*70)

In [ ]:
# Visualization: forecast comparison
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Time series with forecasts
train = forecast_results['train']
test = forecast_results['test']
naive_pred = forecast_results['naive_pred']
ma_pred = forecast_results['ma_pred']

train_idx = np.arange(len(train))
test_idx = np.arange(len(train), len(train) + len(test))

ax = axes[0]
ax.plot(train_idx, train, 'o-', label='Train', alpha=0.7, linewidth=2)
ax.plot(test_idx, test, 'o-', label='Test (actual)', alpha=0.7, linewidth=2)
ax.plot(test_idx, naive_pred, 's--', label='Naive forecast', alpha=0.7, linewidth=2)
ax.plot(test_idx, ma_pred, '^--', label='MA(3) forecast', alpha=0.7, linewidth=2)
ax.axvline(x=len(train)-0.5, color='red', linestyle=':', alpha=0.5, label='Train/test split')
ax.set_xlabel('Time index')
ax.set_ylabel('Value')
ax.set_title('Forecast Comparison: MA(3) vs Naive (Synthetic Series)')
ax.legend(loc='best')
ax.grid(alpha=0.3)

# Plot 2: Error comparison
ax = axes[1]
naive_errors = np.abs(naive_pred - test)
ma_errors = np.abs(ma_pred - test)

x_pos = np.arange(len(test))
ax.bar(x_pos - 0.2, naive_errors, 0.4, label='Naive error', alpha=0.7)
ax.bar(x_pos + 0.2, ma_errors, 0.4, label='MA(3) error', alpha=0.7)
ax.axhline(y=forecast_results['naive_mae'], color='C0', linestyle='--', alpha=0.5, label=f"Naive MAE={forecast_results['naive_mae']:.3f}")
ax.axhline(y=forecast_results['ma_mae'], color='C1', linestyle='--', alpha=0.5, label=f"MA(3) MAE={forecast_results['ma_mae']:.3f}")
ax.set_xlabel('Test point index')
ax.set_ylabel('Absolute error')
ax.set_title('Per-point Forecast Errors')
ax.legend(loc='best')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved as forecast_comparison.png")